In [120]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc, norm
np.random.seed(42) #pseudo random number

# Introduction

For this project, I will use closed form solution, the EM scheme, and Milstein scheme of SDE: $dS/S = rdt + \sigma dW$, to simulate underlying stock price movement paths to find the price for European call/put options and Binary call/put options. 

To go one step beyond, I will also explore and discuss how different schemes and parameters could impact the pricing results. 

According to Exam 2_Resource on Milstein.pdf:

1. Close-form solution follows: $S_{t+\Delta t} = S_{t}e^{(r-\frac{1}{2}\sigma^2)\Delta t+\sigma\phi\sqrt{\Delta t}}$ 
2. EM scheme follows: $S_{t+\Delta t} \sim  S_{t}(1+r\Delta t+\sigma\phi\sqrt{\Delta t})$  
3. Milstein scheme follows: $S_{t+\Delta t} \sim  S_{t}(1+r\Delta t+\sigma\phi\sqrt{\Delta t}+\frac{1}{2}\sigma^2(\phi^2-1)\Delta t)$

where $\phi \sim N(0,1)$

In addition, I have listed several experient sets to measure the impact from different parameters:

Case 1: initial case
For all simulations, I will start with these parameters: (S0=100, r=0.05, sigma=0.2, T=1, n_paths=5000, n_steps=1).  
For all the options, I will start with  these parameters: (S0=100, E=100, T=1, sigma=0.2, r=0.05)

Case 2

Case 2: Case 1 + change n_paths from 50000 to 200000

Case 3: Case 2 + change n_steps from 252 to 2520

Case 4: Case 2 + use Moment Matching variance reduction techniques (Moment Matching)

In [86]:
# Simulation Function for GBM paths with differet methods
def simulate_gbm_paths(S0, r, sigma, T, n_paths, n_steps,scheme='Closed Form Solution'):
    dt = T / n_steps
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = S0

    if scheme == 'Closed Form Solution':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            paths[:, t] = paths[:, t-1] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z) 

    elif scheme == 'Euler Maruyama Scheme':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            paths[:, t] = paths[:, t-1] * (1+ r*dt + sigma * np.sqrt(dt) * z) 

    elif scheme == 'Milstein Scheme':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            paths[:, t] = paths[:, t-1] * (1+ r*dt + sigma * np.sqrt(dt) * z + 1/2*(sigma**2)*(z**2-1)*dt)
            
    return paths

In [149]:
# Simulation Function for GBM paths with differet methods with Moment Matching variance reduction method
def simulate_gbm_paths_moment(S0, r, sigma, T, n_paths, n_steps,scheme='Closed Form Solution'):
    dt = T / n_steps
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = S0

    if scheme == 'Closed Form Solution':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            z = (z - np.mean(z)) / np.std(z)
            paths[:, t] = paths[:, t-1] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z) 

    elif scheme == 'Euler Maruyama Scheme':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            z = (z - np.mean(z)) / np.std(z)
            paths[:, t] = paths[:, t-1] * (1+ r*dt + sigma * np.sqrt(dt) * z) 

    elif scheme == 'Milstein Scheme':
        for t in range(1, n_steps + 1):
            z = np.random.standard_normal(n_paths)
            z = (z - np.mean(z)) / np.std(z)
            paths[:, t] = paths[:, t-1] * (1+ r*dt + sigma * np.sqrt(dt) * z + 1/2*(sigma**2)*(z**2-1)*dt)
            
    return paths

In [140]:
# European Option Price 
def european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction = False,option_type='call',scheme = 'Closed Form Solution'):
    
    if variance_reduction:
        paths = simulate_gbm_paths_moment(S0, r, sigma, T, n_paths,n_steps,scheme=scheme)
    else: 
        paths = simulate_gbm_paths(S0, r, sigma, T, n_paths,n_steps,scheme=scheme)
    
    payoff = np.maximum(paths[:, -1] - K, 0) if option_type == 'call' else np.maximum(K - paths[:, -1], 0) ## calculate payoffs on maturity date
    price = np.exp(-r * T) * np.mean(payoff)   ## take average of all the payoffs at maturity and discount to present
    return price
    
# Binary Option Price 
def binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction = False,option_type='call',scheme = 'Closed Form Solution'):

    if variance_reduction:
        paths = simulate_gbm_paths_moment(S0, r, sigma, T, n_paths,n_steps,scheme=scheme)
    else: 
        paths = simulate_gbm_paths(S0, r, sigma, T, n_paths,n_steps,scheme=scheme)

    payoff = np.maximum(paths[:, -1] - K, 0) if option_type == 'call' else np.maximum(K - paths[:, -1], 0) ## calculate payoffs on maturity date
    payoff[payoff>0] = 1   ## override all the positive payoffs to 1 to get binary payoff for both call and put
    price = np.exp(-r * T) * np.mean(payoff)   ## take average of all the payoffs at maturity and discount to present
    return price

def result_table(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction):
    european_call_price_closed_form = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'call','Closed Form Solution')
    european_put_price_closed_form = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'put','Closed Form Solution')
    european_call_price_EM = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'call','Euler Maruyama Scheme')
    european_put_price_EM = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'put','Euler Maruyama Scheme')
    european_call_price_Milstein = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction,  'call','Milstein Scheme')
    european_put_price_Milstein = european_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction,  'put','Milstein Scheme')
    
    binary_call_price_closed_form = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps, variance_reduction, 'call','Closed Form Solution')
    binary_put_price_closed_form = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'put','Closed Form Solution')
    binary_call_price_EM = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'call','Euler Maruyama Scheme')
    binary_put_price_EM = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,'put','Euler Maruyama Scheme')
    binary_call_price_Milstein = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'call','Milstein Scheme')
    binary_put_price_Milstein = binary_option_price(S0, K, r, sigma, T, n_paths, n_steps,variance_reduction, 'put','Milstein Scheme')
    
    result = pd.DataFrame([[european_call_price_closed_form,european_call_price_EM,european_call_price_Milstein],
                 [european_put_price_closed_form,european_put_price_EM,european_put_price_Milstein],
                 [binary_call_price_closed_form,binary_call_price_EM,binary_call_price_Milstein],
                 [binary_call_price_closed_form,binary_call_price_EM,binary_call_price_Milstein]],
                 index = pd.MultiIndex.from_tuples([('European', 'Call'), ('European', 'Put'),('Binary', 'Call'), ('Binary', 'Put')], names=('Option Type', 'Call/Put')),
                 columns = ['Closed Form','Euler Maruyama','Milstein'])

    result['Closed Form - EM'] = result['Closed Form'] - result['Euler Maruyama'] 
    result['Closed Form - Milstein'] = result['Closed Form'] - result['Milstein'] 

    return result

# Results

## Case 1: initial case

In [164]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=1, n_paths=5000, n_steps=1,variance_reduction=False)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        10.352051       10.109885  10.170907   
            Put          5.500687        5.333533   5.355497   
Binary      Call         0.542772        0.585196   0.540869   
            Put          0.542772        0.585196   0.540869   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.242166                0.181143  
            Put               0.167154                0.145189  
Binary      Call             -0.042425                0.001902  
            Put              -0.042425                0.001902

## Case 2: Case 1 + n_paths change from 5000 to 50000

In [165]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=1, n_paths=50000, n_steps=1,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        10.457508       10.205450  10.057544   
            Put          5.575982        5.444351   5.310901   
Binary      Call         0.532879        0.570338   0.533754   
            Put          0.532879        0.570338   0.533754   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.252058                0.399964  
            Put               0.131631                0.265082  
Binary      Call             -0.037459               -0.000875  
            Put              -0.037459               -0.000875

## Case 3: Case 1 + change n_steps from 1 to 252


In [167]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=1, n_paths=5000, n_steps=252,variance_reduction=False)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        10.607065       10.426772  10.527238   
            Put          5.668633        5.351485   5.518796   
Binary      Call         0.542201        0.533830   0.537064   
            Put          0.542201        0.533830   0.537064   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.180293                0.079828  
            Put               0.317148                0.149837  
Binary      Call              0.008371                0.005137  
            Put               0.008371                0.005137

## Case 4: Case 1 + use Moment Matching variance reduction techniques

In [168]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=1, n_paths=5000, n_steps=1,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        10.470736       10.247296  10.067895   
            Put          5.569492        5.471800   5.293345   
Binary      Call         0.538396        0.569596   0.533640   
            Put          0.538396        0.569596   0.533640   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.223440                0.402841  
            Put               0.097692                0.276147  
Binary      Call             -0.031200                0.004756  
            Put              -0.031200                0.004756

## Case 5: Case 2 + Case 3 + Case 4

In [173]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=1, n_paths=50000, n_steps=252,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        10.426175       10.475867  10.525417   
            Put          5.572377        5.555641   5.556788   
Binary      Call         0.530995        0.532612   0.531147   
            Put          0.530995        0.532612   0.531147   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call             -0.049692               -0.099242  
            Put               0.016736                0.015589  
Binary      Call             -0.001617               -0.000152  
            Put              -0.001617               -0.000152

## Case 6: Case 5 + change S from 100 to 80 (OTM)

In [174]:
result_table(S0=80, K=100, r=0.05, sigma=0.2, T=1, n_paths=50000, n_steps=252,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call         1.854113        1.812670   1.883340   
            Put         16.981592       16.983847  16.976152   
Binary      Call         0.159768        0.159959   0.160377   
            Put          0.159768        0.159959   0.160377   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.041442               -0.029228  
            Put              -0.002255                0.005440  
Binary      Call             -0.000190               -0.000609  
            Put              -0.000190               -0.000609

## Case 7: Case 5 + change r from 0.05 to 0.01

In [175]:
result_table(S0=100, K=100, r=0.01, sigma=0.2, T=1, n_paths=50000, n_steps=252,variance_reduction=True)

Closed Form  Euler Maruyama  Milstein  Closed Form - EM  \
Option Type Call/Put                                                            
European    Call         8.423677        8.434971  8.500412         -0.011293   
            Put          7.507736        7.399276  7.453477          0.108460   
Binary      Call         0.475739        0.477204  0.475165         -0.001465   
            Put          0.475739        0.477204  0.475165         -0.001465   

                      Closed Form - Milstein  
Option Type Call/Put                          
European    Call                   -0.076734  
            Put                     0.054259  
Binary      Call                    0.000574  
            Put                     0.000574

## Case 8: Case 5 + change sigma from 0.2 to 0.3

In [177]:
result_table(S0=100, K=100, r=0.05, sigma=0.3, T=1, n_paths=50000, n_steps=252,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        14.255318       14.190961  14.289936   
            Put          9.349447        9.376032   9.359075   
Binary      Call         0.481455        0.483129   0.481398   
            Put          0.481455        0.483129   0.481398   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call              0.064357               -0.034618  
            Put              -0.026585               -0.009628  
Binary      Call             -0.001674                0.000057  
            Put              -0.001674                0.000057

## Case 9: Case 5 + change T from 1 to 3

In [178]:
result_table(S0=100, K=100, r=0.05, sigma=0.2, T=3, n_paths=50000, n_steps=252,variance_reduction=True)

Closed Form  Euler Maruyama   Milstein  \
Option Type Call/Put                                           
European    Call        20.954861       20.980810  20.938871   
            Put          7.027035        7.006367   6.973590   
Binary      Call         0.518129        0.519954   0.517526   
            Put          0.518129        0.519954   0.517526   

                      Closed Form - EM  Closed Form - Milstein  
Option Type Call/Put                                            
European    Call             -0.025949                0.015990  
            Put               0.020668                0.053444  
Binary      Call             -0.001825                0.000602  
            Put              -0.001825                0.000602

## Error Analysis:

According to Exam 2_Resource on Milstein.pdf, Euler Maruyama scheme has an error of $O(\Delta t^{0.5})$, while through Taylor expansion of the exact solution, Milstein adds the second order term, the Milstein correction $1/2(\phi^2-1)\Delta t$, leading to an error of $O(\Delta t)$ and increasing the appproximation accuracy. In other words, Euler Maruyama has a strong convergence order of 0.5 and Milstein has a strong convergence order of 1. This means Milstein simulates the GBM better.

# Observations
1. For European, put price is lower than call price for all methods when at the money. This is due to the interest rate/cost of carry, which can also be illustrated through the put call parity: $C-P=K(1-e^{-rT})\$, where both K and r are positive. [Refer the link for more information.](https://www.thebluecollarinvestor.com/why-at-the-money-calls-are-frequently-priced-higher-than-at-the-money-puts-2/)

2. For Binary, call and put option have the same price when at the money. When at the money, the likelihood of the stock price ending up above the strike price is roughly equal to the likelihood of it ending up below the strike price in the risk-neutral world. 

3. Binary call/put option prices are much cheaper than European call/put prices with the same parameters. This is because usually binary options are much cheaper as they cap on the maximum possible profit and will not benefit from the massive rally relative to the stock price.

4. While Milstein method provides a more accurate approximation, compared to Euler Maruyama method, it does NOT really improve the expected payoff for pricing! This is because for pricing a European option using Monte Carlo simulation, we only care about the expected value of the final payoff. We do not need the paths to be accurate in a pathwise sense. Both the Euler and Milstein schemes have a weak convergence order of 1 for a smooth payoff function. [This paper explained the difference between strong and weak convergence for EU and Milstein](http://www.columbia.edu/~mh2078/MonteCarlo/MCS_SDEs.pdf)

# Conclusions
1. When comparing Case 1 and Case 2, we can conclude that increasing the number of simulated paths can improve the pricing accuracy. The total error is composed of two independent parts: statistical error and discretization error. This lowered the stochastic error.

2. When comparing Case 2 and Case 3, we can conclude that increasing number of steps in each path will NOT necessarily improve the pricing accuracy. Althrough increasing the number of steps lower the discretization error, and it may also introduce other numerical issues, such as floating-point precision issue. Computers use a finite number of bits to represent real numbers, and if a very small number is added to a much larger number, the small number may be lost or rounded off due to a loss of precision. This effect can become more pronounced as the number of steps increases, offsetting any theoretical benefit. [To learn more about the floating-point precision issue.](https://learn.microsoft.com/en-us/cpp/build/why-floating-point-numbers-may-lose-precision?view=msvc-170)

3. When comparing Case 2 and Case 4, we can conclude that using variance reduction techniques, such as moment matching, can efficiently increase the pricing accuracy. The moment matching involves adjusting a model so that its moments—such as mean, variance, skewness, and kurtosis—match those of the observed data. Here we adjust random generated z so that they are exactly the standard normal distribution. [Introduction to Moment Matching Method.](https://statisticseasily.com/glossario/what-is-moment-matching-detailed-explanation/)

4. For European, put price is lower than call price for all methods when at the money.

5. For Binary, call and put option have the same price when at the money.

6. Binary call/put option prices are much cheaper than European call/put prices with the same parameters.

7. While Milstein method provides a more accurate approximation, compared to Euler Maruyama method, it does NOT really improve the expected payoff for pricing.

# Reference

References are all added at the end of the comparisons and observations in form on links